In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
import pandas as pd


cities = ["https://www.talabat.com/egypt/restaurants/7766/downtown-magra-el-ouyon", "https://www.talabat.com/egypt/restaurants/7177/el-hadara"]
today = datetime.now().strftime("%Y-%m-%d")

for city in cities:
    print(f"Running for {city}...")

    # Setup Chrome options
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)

    # === STEP 1: GET ALL RESTAURANT LINKS ===
    base_url =city
    all_restaurant_links = []

    # Loop through all pages
    for page_num in range(1, 57):  # Change to 57 for all pages
        try:
            # Construct URL with page parameter
            if page_num == 1:
                url = base_url
            else:
                url = f"{base_url}?page={page_num}"

            print(f"Scraping page {page_num}...")
            driver.get(url)

            # Wait for page to load
            time.sleep(5)

            # Parse HTML
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            # Find all restaurant links
            restaurant_links = soup.find_all("a", {"data-testid": "restaurant-a"})

            for link in restaurant_links:
                href = link.get("href")
                if href:
                    full_url = f"https://www.talabat.com{href}" if href.startswith("/") else href
                    all_restaurant_links.append(full_url)

            print(f"Found {len(restaurant_links)} restaurants on page {page_num}")

        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            continue

    print(f"\nTotal restaurants found: {len(all_restaurant_links)}")
    print(f"Unique restaurants: {len(set(all_restaurant_links))}")

    # === STEP 2: SCRAPE EACH RESTAURANT'S MENU ===
    all_menu_items = []
    all_restaurant_info = []

    for idx, restaurant_url in enumerate(set(all_restaurant_links), 1):
        print(f"\nScraping restaurant {idx}/{len(set(all_restaurant_links))}: {restaurant_url}")

        try:
            driver.get(restaurant_url)
            time.sleep(5)

            # Scroll to load all menu items
            last_height = driver.execute_script("return document.body.scrollHeight")
            while True:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2)
                new_height = driver.execute_script("return document.body.scrollHeight")
                if new_height == last_height:
                    break
                last_height = new_height

            # Parse HTML
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")

            # === EXTRACT RESTAURANT INFORMATION ===

            # Get restaurant name and location from h1 tag
            restaurant_title_elem = soup.find("h1", {"data-testid": "restaurant-title"})
            if restaurant_title_elem:
                # Extract the main text (restaurant name)
                restaurant_name = restaurant_title_elem.contents[0].strip() if restaurant_title_elem.contents else "Unknown"

                # Extract location from small tag
                location_elem = restaurant_title_elem.find("small")
                if location_elem:
                    location_text = location_elem.get_text(strip=True)
                    # Remove "in" from the beginning if present
                    location = location_text.replace("in ", "", 1) if location_text.startswith("in ") else location_text
                else:
                    location = "Location not found"
            else:
                # Fallback to simple h1 if data-testid not found
                restaurant_name_elem = soup.find("h1")
                restaurant_name = restaurant_name_elem.get_text(strip=True) if restaurant_name_elem else "Unknown"
                location = "Location not found"

            # Get cuisines
            cuisines_elem = soup.find("p", {"data-testid": "cuisines"})
            cuisines = cuisines_elem.get_text(strip=True) if cuisines_elem else "Cuisines not found"

            # List to store all prices for this restaurant
            restaurant_prices = []

            # Find menu items with both class variations
            items = soup.find_all("div", {"class": lambda c: c and (
                "sc-a31f9fb2-0 dyJtfK d-flex justify-content-between py-2 clickable" in c or
                "sc-a31f9fb2-0 eQGrrN d-flex justify-content-between py-2 clickable" in c or
                "d-flex justify-content-between py-2 clickable" in c  # More flexible matching
            )})

            print(f"Found {len(items)} menu items")

            for item in items:
                # Extract item details
                name_elem = item.find("div", {"class": lambda c: c and "f-15" in c})
                desc_elem = item.find("div", {"class": lambda c: c and "f-12 description" in c})
                price_elem = item.find("span", {"class": "currency"})

                # Find image
                img_container = item.find("div", {"class": lambda c: c and "ecdIEH" in c})
                img_elem = img_container.find("img") if img_container else None

                # If img_container not found with that class, try to find img directly
                if not img_elem:
                    img_elem = item.find("img")

                # Get text values
                name = name_elem.get_text(strip=True) if name_elem else None
                desc = desc_elem.get_text(strip=True) if desc_elem else None
                price = price_elem.get_text(strip=True) if price_elem else None
                img_url = img_elem["src"] if img_elem and img_elem.has_attr("src") else None

                # Add price to the restaurant's price list (only the number)
                if price:
                    # Extract only numbers from price (remove currency symbols, text, etc.)
                    price_numbers = re.findall(r'[\d.]+', price)
                    if price_numbers:
                        restaurant_prices.append(float(price_numbers[0]))

                if name:
                    all_menu_items.append({
                        "Restaurant": restaurant_name,
                        "Restaurant_URL": restaurant_url,
                        "Location": location,
                        "Cuisines": cuisines,
                        "Item_Name": name,
                        "Description": desc,
                        "Price": price,
                        "Image_URL": img_url
                    })

            # Store restaurant summary information
            all_restaurant_info.append({
                "Restaurant": restaurant_name,
                "Location": location,
                "Cuisines": cuisines,
                "URL": restaurant_url,
                "Total_Items": len(items),
                "Prices_List": restaurant_prices,
                "Min_Price": min(restaurant_prices) if restaurant_prices else None,
                "Max_Price": max(restaurant_prices) if restaurant_prices else None,
                "Avg_Price": sum(restaurant_prices)/len(restaurant_prices) if restaurant_prices else None
            })

            print(f"Restaurant: {restaurant_name}")
            print(f"Location: {location}")
            print(f"Cuisines: {cuisines}")
            print(f"Number of prices collected: {len(restaurant_prices)}")

        except Exception as e:
            print(f"Error scraping {restaurant_url}: {e}")
            continue

    # Close driver
    driver.quit()

    # === DISPLAY FINAL RESULTS ===

    # Create DataFrames
    final_df = pd.DataFrame(all_menu_items)
    restaurant_df = pd.DataFrame(all_restaurant_info)

    print(f"\nTotal menu items scraped: {len(final_df)}")
    print(f"From {final_df['Restaurant'].nunique() if not final_df.empty else 0} restaurants")

    print("\nSample menu items:")
    if not final_df.empty:
        print(final_df.head(10))

    print("\nRestaurant Summary:")
    if not restaurant_df.empty:
        print(restaurant_df[['Restaurant', 'Location', 'Cuisines', 'Total_Items', 'Min_Price', 'Max_Price', 'Avg_Price']].head(10))

    # Create a simplified view with just restaurant, location, cuisines, and prices list
    simple_restaurant_list = []
    for rest in all_restaurant_info:
        simple_restaurant_list.append({
            "Restaurant": rest['Restaurant'],
            "Location": rest['Location'],
            "Cuisines": rest['Cuisines'],
            "All_Prices": rest['Prices_List']  # This is the list of all prices without dish names
        })

    simple_df = pd.DataFrame(simple_restaurant_list)
    if city == "https://www.talabat.com/egypt/restaurants/7177/el-hadara":
      city_name = "alex"
    else:
      city_name = "cairo"
    file_name = f"{city_name}_{today}_talabat.csv"
    restaurant_df.to_csv(f'D:\Mashro3 El T5arog\pipeline\scraping_scripts\Data_files\{file_name}', index=False)

    print("\nSimple Restaurant List with Prices:")
    if not simple_df.empty:
        for idx, row in simple_df.head(5).iterrows():
            print(f"\nRestaurant: {row['Restaurant']}")
            print(f"Location: {row['Location']}")
            print(f"Cuisines: {row['Cuisines']}")
            print(f"Prices: {row['All_Prices'][:10] if len(row['All_Prices']) > 10 else row['All_Prices']}")

    # Display statistics
    print("\n=== OVERALL STATISTICS ===")
    if all_restaurant_info:
        total_restaurants = len(all_restaurant_info)
        total_items = sum(r['Total_Items'] for r in all_restaurant_info)
        all_prices_flat = [price for r in all_restaurant_info for price in r['Prices_List']]

        print(f"Total restaurants scraped: {total_restaurants}")
        print(f"Total menu items found: {total_items}")
        if all_prices_flat:
            print(f"Overall price range: {min(all_prices_flat)} - {max(all_prices_flat)}")
            print(f"Overall average price: {sum(all_prices_flat)/len(all_prices_flat):.2f}")

Running for https://www.talabat.com/egypt/restaurants/7766/downtown-magra-el-ouyon...
Scraping page 1...
Found 15 restaurants on page 1
Scraping page 2...
Found 15 restaurants on page 2

Total restaurants found: 30
Unique restaurants: 30

Scraping restaurant 1/30: https://www.talabat.com/egypt/restaurant/514551/mince-giza?aid=7766
Found 86 menu items
Restaurant: Mince
Location: inGiza,Egypt
Cuisines: Burgers, Fast Food, American
Number of prices collected: 79

Scraping restaurant 2/30: https://www.talabat.com/egypt/restaurant/649438/kyoto-sushi-dokki--mosaddak?aid=7766
Found 238 menu items
Restaurant: Kyoto Sushi
Location: inDokki - Mosaddak,Egypt
Cuisines: Sushi, Japanese , Asian
Number of prices collected: 192

Scraping restaurant 3/30: https://www.talabat.com/egypt/restaurant/502351/momen--al-manial?aid=7766
Found 112 menu items
Restaurant: Mo'men
Location: inAl Manial,Egypt
Cuisines: Sandwiches, Fish & Seafood, Burgers, Chicken
Number of prices collected: 85

Scraping restaurant 4/